# jev-benchmark

面向语义决策模型的中文言下之意评测集（100 题 Choice）。本 notebook：**① 选择模型并运行评测 → ② 查看排行榜与逐题报表 → 附：本地模型三种决策原语演示**。模型与指标说明见 [README](README.md)。

准备：`pip install -r requirements.txt`；评测本地模型需先运行 `python -m jev_benchmark.download`；评测远程模型需在 `.env` 中配置密钥（参考 `.env.example`）。

## 1. 运行评测

在 `MODELS` 中选择参评模型，可多选同题对比：

| 键 | 模型 | 说明 |
|---|---|---|
| `"local"` | Qwen3.5-0.8B-Jev | 本地 ONNX，CPU 约 3 分钟，不联网 |
| `"jev"` | Jev-API | 官方托管，100 次请求，**可能计费** |
| `"intranet"` | 内网 Jev-like 服务 | 如 Qwen3-1.7B-Jev，需自行部署 |

结果写入 `results/<模型组合>.json`，运行结束后直接显示报表。

In [ ]:
import json
from IPython.display import HTML, display
from jev_benchmark import DATA_PATH
from jev_benchmark.evaluate import load_contestant, render_report, result_path, run_benchmark

MODELS = ["local"]

contestants = [load_contestant(key) for key in MODELS]
output_path = result_path(MODELS)
report = run_benchmark(DATA_PATH, contestants, output_path,
                       lambda done, total: print(f"\r已完成 {done}/{total}", end=""))
print(f"\n已保存：results/{output_path.name}")
display(HTML(render_report(report, json.loads(DATA_PATH.read_text(encoding="utf-8")))))

## 2. 查看已保存结果

不调用任何模型，可直接运行。排行榜汇总 `results/` 中与当前题集哈希一致的完整结果（同一模型取最近一次）；`REPORT` 指定要展开的逐题报表，仓库自带 `local-jev.json`。

In [ ]:
import json
from IPython.display import HTML, display
from jev_benchmark import DATA_PATH
from jev_benchmark.evaluate import load_results, render_leaderboard, render_report

REPORT = "local-jev.json"

dataset = json.loads(DATA_PATH.read_text(encoding="utf-8"))
saved = load_results()
display(HTML(render_leaderboard(saved, dataset)))
if REPORT in saved:
    display(HTML(render_report(saved[REPORT], dataset)))
else:
    print(f"未找到 {REPORT}；可选：{', '.join(saved) or '无'}")

## 附：本地模型的三种决策原语

`noul`（是/否）、`choice`（多选一）、`score`（有序评分）。本地模型读取首个生成位置的选项字母 logits 并仅在候选项间归一化，概率未经校准。

In [ ]:
from jev_benchmark.local_model import LocalModel

# 复用第 1 节已加载的本地模型，避免重复占用内存
local = next((c for c in globals().get("contestants", []) if c.key == "local"), None) or LocalModel()

urgency = local.score("救命！我的付款已经连续 3 天失败了。我今天必须拿到这笔钱。", "这条消息是否表达了紧迫性？",
                      {"A": "否", "B": "是"})
team = local.score("同一笔订单向我收了两次款。请退还重复支付的金额。", "哪个团队应该处理这项客户请求？",
                   {"A": "账单团队 — 付款与退款", "B": "配送团队 — 配送问题", "C": "技术团队 — 产品故障"})
severity = local.score("这个应用今天已经崩溃五次了。我无法完成工作，感到非常沮丧。", "客户的沮丧程度有多严重？",
                       {"A": "0 — 无", "B": "1 — 轻微", "C": "2 — 中等", "D": "3 — 严重"})

print("noul   紧迫性（A=否 / B=是）：", {k: round(v, 3) for k, v in urgency.items()})
print("choice 处理团队：", {k: round(v, 3) for k, v in team.items()})
print(f"score  沮丧程度期望值（0–3）：{sum(i * p for i, p in enumerate(severity.values())):.2f}")